# Mixture Density Networks

Right now our Worker outputs a single `(dx, dy)` prediction at each step. This is a problem: sketches are inherently ambiguous. Given a partial drawing, there are many plausible ways to continue -- not just one.

A **Mixture Density Network (MDN)** replaces the single output with a mixture of Gaussians. Instead of predicting one `(dx, dy)`, it predicts K Gaussian components, each with its own mean, variance, and mixing weight. We then sample from this mixture to get diverse, creative outputs.

This is **Checkpoint 4: Creative Diversity**.

## 1. What is a Gaussian mixture?

A mixture of K Gaussians defines a distribution:
```
p(x) = sum_k pi_k * N(x | mu_k, sigma_k)
```
where:
- `pi_k` : mixing weights (sum to 1)
- `mu_k`  : mean of component k
- `sigma_k`: standard deviation of component k

The MDN outputs all of these parameters. We then sample from the mixture.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Visualise what a mixture of Gaussians looks like
x = np.linspace(-5, 5, 1000)

# Two component mixture
pi    = [0.6, 0.4]
mu    = [-2.0, 2.0]
sigma = [0.5, 1.0]

pdf = sum(
    pi[k] * (1 / (sigma[k] * np.sqrt(2 * np.pi)))
    * np.exp(-0.5 * ((x - mu[k]) / sigma[k]) ** 2)
    for k in range(2)
)

plt.figure(figsize=(8, 4))
plt.plot(x, pdf, color='#534AB7', linewidth=2)
plt.fill_between(x, pdf, alpha=0.2, color='#534AB7')
plt.title('Mixture of 2 Gaussians')
plt.xlabel('x'); plt.ylabel('p(x)')
plt.tight_layout(); plt.show()


## 2. The MDN output head

We replace the final linear layer of the Worker with an MDN head.
For K mixture components and 2D output (dx, dy), the head predicts:
- `pi`    : K mixing weights
- `mu`    : K x 2 means
- `sigma` : K x 2 standard deviations (positive)
- `rho`   : K correlation coefficients between dx and dy (between -1 and 1)

In [ ]:
class MDNHead(nn.Module):
    def __init__(self, input_dim, num_mixtures=20):
        '''
        input_dim    : size of the LSTM output
        num_mixtures : number of Gaussian components K
        '''
        super().__init__()
        self.K = num_mixtures
        # Predict: pi (K), mu_x (K), mu_y (K), sigma_x (K), sigma_y (K), rho (K)
        self.fc = nn.Linear(input_dim, num_mixtures * 6)

    def forward(self, h):
        '''
        h : LSTM hidden output, shape (batch, seq_len, input_dim)
        Returns a dict of mixture parameters, each shape (batch, seq_len, K)
        '''
        out = self.fc(h)   # (batch, seq_len, K*6)
        K   = self.K

        pi    = F.softmax(out[..., :K],         dim=-1)            # mixing weights sum to 1
        mu_x  = out[..., K:2*K]
        mu_y  = out[..., 2*K:3*K]
        sig_x = torch.exp(out[..., 3*K:4*K])                      # must be positive
        sig_y = torch.exp(out[..., 4*K:5*K])
        rho   = torch.tanh(out[..., 5*K:6*K])                     # between -1 and 1

        return {'pi': pi, 'mu_x': mu_x, 'mu_y': mu_y,
                'sig_x': sig_x, 'sig_y': sig_y, 'rho': rho}

# Quick test
head = MDNHead(input_dim=512, num_mixtures=20)
dummy_h = torch.randn(4, 100, 512)   # (batch, seq_len, worker_dim)
params = head(dummy_h)
print('pi shape   :', params['pi'].shape)    # (4, 100, 20)
print('mu_x shape :', params['mu_x'].shape)


## 3. Sampling from the mixture

At inference time we:
1. Sample a component index k from the mixing weights pi
2. Sample (dx, dy) from the bivariate Gaussian of component k

A **temperature** parameter controls creativity:
- Low temperature: model picks the most likely stroke (conservative)
- High temperature: model samples more randomly (creative but noisier)

In [ ]:
def sample_mdn(params, temperature=1.0):
    '''
    Sample (dx, dy) from the mixture at each step.

    params      : dict of mixture parameters, each (batch, seq_len, K)
    temperature : controls randomness (0.1 = conservative, 1.0 = normal, 2.0 = wild)

    Returns:
        dx, dy : sampled positions, each shape (batch, seq_len)
    '''
    pi    = params['pi']    / temperature
    pi    = pi / pi.sum(dim=-1, keepdim=True)   # renormalise after temperature scaling
    mu_x  = params['mu_x']
    mu_y  = params['mu_y']
    sig_x = params['sig_x'] * temperature
    sig_y = params['sig_y'] * temperature
    rho   = params['rho']

    # Sample component index for each step
    batch, seq_len, K = pi.shape
    pi_flat = pi.reshape(-1, K)
    k_idx   = torch.multinomial(pi_flat, 1).squeeze(-1)   # (batch * seq_len,)
    k_idx   = k_idx.reshape(batch, seq_len)

    # Gather the parameters of the chosen component
    k_idx_exp = k_idx.unsqueeze(-1)
    mu_x_sel  = mu_x.gather(-1, k_idx_exp).squeeze(-1)
    mu_y_sel  = mu_y.gather(-1, k_idx_exp).squeeze(-1)
    sx        = sig_x.gather(-1, k_idx_exp).squeeze(-1)
    sy        = sig_y.gather(-1, k_idx_exp).squeeze(-1)
    r         = rho.gather(-1, k_idx_exp).squeeze(-1)

    # Sample from bivariate Gaussian
    eps_x = torch.randn_like(mu_x_sel)
    eps_y = torch.randn_like(mu_y_sel)
    dx = mu_x_sel + sx * eps_x
    dy = mu_y_sel + sy * (r * eps_x + torch.sqrt(1 - r**2) * eps_y)

    return dx, dy

# Test sampling
dx, dy = sample_mdn(params, temperature=1.0)
print('dx shape:', dx.shape)   # (4, 100)


## 4. The MDN loss

Instead of MSE, we use the **negative log-likelihood** of the true (dx, dy) under the predicted mixture. This trains the model to assign high probability to the correct output.

In [ ]:
def mdn_loss(params, dx_true, dy_true):
    '''
    Negative log-likelihood loss for the MDN.

    params : dict of mixture parameters, each (batch, seq_len, K)
    dx_true, dy_true : ground truth deltas (batch, seq_len)
    '''
    mu_x  = params['mu_x']
    mu_y  = params['mu_y']
    sig_x = params['sig_x']
    sig_y = params['sig_y']
    rho   = params['rho']
    pi    = params['pi']

    dx = dx_true.unsqueeze(-1)   # (batch, seq_len, 1) for broadcasting
    dy = dy_true.unsqueeze(-1)

    # Normalised deviations
    z_x = (dx - mu_x) / (sig_x + 1e-8)
    z_y = (dy - mu_y) / (sig_y + 1e-8)

    # Bivariate Gaussian exponent
    z = z_x**2 + z_y**2 - 2 * rho * z_x * z_y
    denom = 1 - rho**2 + 1e-8
    exponent = -z / (2 * denom)

    # Gaussian PDF for each component
    norm = 1.0 / (2 * np.pi * sig_x * sig_y * torch.sqrt(denom))
    gauss = norm * torch.exp(exponent)

    # Weighted sum over components
    likelihood = (pi * gauss).sum(dim=-1)   # (batch, seq_len)
    nll = -torch.log(likelihood + 1e-8).mean()
    return nll

# Test
dx_true = torch.randn(4, 100)
dy_true = torch.randn(4, 100)
loss = mdn_loss(params, dx_true, dy_true)
print('MDN NLL loss:', loss.item())


## 5. Full model with MDN

In [ ]:
import json, os, urllib.request
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

def drawing_to_stroke5(drawing, max_len=200):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            if i < len(xs) - 1: p1, p2, p3 = 1, 0, 0
            else: p1, p2, p3 = 0, 1, 0
            strokes.append([dx, dy, p1, p2, p3])
    strokes.append([0, 0, 0, 0, 1])
    s5 = np.array(strokes, dtype=np.float32)
    if len(s5) > max_len:
        s5 = s5[:max_len]; s5[-1] = [0, 0, 0, 0, 1]
    return s5

def normalise_stroke5(stroke5):
    s = stroke5.copy(); coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d = json.loads(line)
                s5 = drawing_to_stroke5(d['drawing'], max_len=max_len)
                s5 = normalise_stroke5(s5)
                self.samples.append(torch.tensor(s5, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    lengths = [s.shape[0] for s in batch]
    return pad_sequence(batch, batch_first=True, padding_value=0.0), lengths

os.makedirs('data', exist_ok=True)
path = 'data/cat.ndjson'
if not os.path.exists(path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson', path)

dataset = QuickDrawDataset(path, max_len=200, max_samples=3000)
loader  = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.lstm      = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_mu     = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)
    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[0], h[1]], dim=-1)
        return self.fc_mu(h), self.fc_logvar(h)

def reparameterise(mu, logvar):
    return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

class Conductor(nn.Module):
    def __init__(self, latent_dim=128, conductor_dim=512, output_dim=256, num_segments=16):
        super().__init__()
        self.num_segments = num_segments
        self.output_dim   = output_dim
        self.fc_init = nn.Linear(latent_dim, conductor_dim)
        self.lstm    = nn.LSTM(output_dim, conductor_dim, batch_first=True)
        self.fc_out  = nn.Linear(conductor_dim, output_dim)
    def forward(self, z):
        batch = z.shape[0]
        h = torch.tanh(self.fc_init(z)).unsqueeze(0)
        c = torch.zeros_like(h)
        inp = torch.zeros(batch, 1, self.output_dim, device=z.device)
        outs = []
        for _ in range(self.num_segments):
            out, (h, c) = self.lstm(inp, (h, c))
            cv = self.fc_out(out); outs.append(cv); inp = cv
        return torch.cat(outs, dim=1)

class WorkerMDN(nn.Module):
    def __init__(self, input_dim=5, conductor_out_dim=256,
                 worker_dim=512, num_mixtures=20, num_pen_states=3):
        super().__init__()
        self.lstm    = nn.LSTM(input_dim + conductor_out_dim, worker_dim, batch_first=True)
        self.mdn     = MDNHead(worker_dim, num_mixtures)
        self.fc_pen  = nn.Linear(worker_dim, num_pen_states)   # separate head for pen state

    def forward(self, stroke_seq, c_seq, segment_len):
        batch, seq_len, _ = stroke_seq.shape
        c_exp = c_seq.repeat_interleave(segment_len, dim=1)[:, :seq_len, :]
        worker_in = torch.cat([stroke_seq, c_exp], dim=-1)
        h, _ = self.lstm(worker_in)
        mdn_params = self.mdn(h)
        pen_logits = self.fc_pen(h)   # (batch, seq_len, 3)
        return mdn_params, pen_logits

class HierarchicalMDNVAE(nn.Module):
    def __init__(self, input_dim=5, latent_dim=128, num_segments=16, num_mixtures=20):
        super().__init__()
        self.num_segments = num_segments
        self.encoder   = Encoder(input_dim, 256, latent_dim)
        self.conductor = Conductor(latent_dim, 512, 256, num_segments)
        self.worker    = WorkerMDN(input_dim, 256, 512, num_mixtures)

    def forward(self, x, lengths):
        mu, logvar = self.encoder(x, lengths)
        z          = reparameterise(mu, logvar)
        c_seq      = self.conductor(z)
        start      = torch.zeros(x.shape[0], 1, x.shape[2], device=x.device)
        dec_in     = torch.cat([start, x[:, :-1, :]], dim=1)
        seg_len    = x.shape[1] // self.num_segments + 1
        mdn_params, pen_logits = self.worker(dec_in, c_seq, seg_len)
        return mdn_params, pen_logits, mu, logvar

model = HierarchicalMDNVAE()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

padded, lengths = next(iter(loader))
mdn_params, pen_logits, mu, logvar = model(padded, lengths)
print('pi shape      :', mdn_params['pi'].shape)
print('pen_logits    :', pen_logits.shape)


## 6. Training the full model

In [ ]:
def full_loss(mdn_params, pen_logits, target, mu, logvar, kl_weight=0.5):
    dx_true = target[:, :, 0]
    dy_true = target[:, :, 1]
    pen_true = target[:, :, 2:].argmax(dim=-1).long()

    nll  = mdn_loss(mdn_params, dx_true, dy_true)
    pen  = F.cross_entropy(pen_logits.reshape(-1, 3), pen_true.reshape(-1))
    kl   = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return nll + pen + kl_weight * kl, nll, pen, kl


In [ ]:
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = HierarchicalMDNVAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
history    = {'total': [], 'nll': [], 'pen': [], 'kl': []}

for epoch in range(num_epochs):
    model.train()
    totals   = {k: 0.0 for k in history}
    kl_weight = min(0.5, epoch / num_epochs)

    for padded, lengths in loader:
        padded = padded.to(device)
        mdn_params, pen_logits, mu, logvar = model(padded, lengths)
        loss, nll, pen, kl = full_loss(mdn_params, pen_logits, padded, mu, logvar, kl_weight)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        totals['total'] += loss.item(); totals['nll'] += nll.item()
        totals['pen']   += pen.item();  totals['kl']  += kl.item()

    n = len(loader)
    for k in history: history[k].append(totals[k] / n)
    print(f'Epoch {epoch+1:02d} | loss {history["total"][-1]:.4f} | '
          f'nll {history["nll"][-1]:.4f} | kl {history["kl"][-1]:.4f}')

torch.save(model.state_dict(), 'hierarchical_mdn_vae.pth')
print('Model saved.')


## 7. Sampling diverse completions

Now the fun part -- given the same starting strokes, generate multiple different completions by sampling from the MDN.

In [ ]:
def generate_completion(model, context, num_completions=4, max_len=150, temperature=0.8):
    '''
    Given a partial sketch (context), generate multiple completions.

    context         : tensor shape (1, context_len, 5)
    num_completions : how many diverse completions to generate
    temperature     : controls diversity
    '''
    model.eval()
    completions = []

    with torch.no_grad():
        for _ in range(num_completions):
            mu, logvar = model.encoder(context, [context.shape[1]])
            z = reparameterise(mu, logvar)   # each sample gives a slightly different z
            c_seq = model.conductor(z)

            # Start autoregressive generation from the last context step
            h, c_cell = None, None
            input_step = context[:, -1:, :]   # last known step
            generated = []

            seg_len = max_len // model.num_segments + 1
            c_exp   = c_seq.repeat_interleave(seg_len, dim=1)[:, :max_len, :]

            for t in range(max_len):
                c_t      = c_exp[:, t:t+1, :]
                worker_in = torch.cat([input_step, c_t], dim=-1)

                if h is None:
                    lstm_out, (h, c_cell) = model.worker.lstm(worker_in)
                else:
                    lstm_out, (h, c_cell) = model.worker.lstm(worker_in, (h, c_cell))

                mdn_p    = model.worker.mdn(lstm_out)
                pen_log  = model.worker.fc_pen(lstm_out)

                dx, dy   = sample_mdn(mdn_p, temperature=temperature)
                pen_state = F.softmax(pen_log.squeeze(1), dim=-1)
                pen_idx   = torch.multinomial(pen_state, 1)   # sample pen state
                pen_onehot = F.one_hot(pen_idx.squeeze(-1), num_classes=3).float()

                step = torch.cat([dx, dy, pen_onehot], dim=-1).unsqueeze(1)
                generated.append(step)
                input_step = step

                # Stop if EOS (p3=1)
                if pen_onehot[0, 2] == 1:
                    break

            completions.append(torch.cat(generated, dim=1).squeeze(0).cpu().numpy())

    return completions

# Test on one sketch
sample_sketch = dataset[0].unsqueeze(0).to(device)
context_len   = sample_sketch.shape[1] // 3   # use first third as context
context       = sample_sketch[:, :context_len, :]

completions = generate_completion(model, context, num_completions=4, temperature=0.8)
print(f'Generated {len(completions)} completions')
print(f'Completion 0 length: {completions[0].shape[0]} steps')


In [ ]:
def stroke5_to_absolute(stroke5):
    abs_coords = np.cumsum(stroke5[:, :2], axis=0)
    pen_up = (stroke5[:, 3] + stroke5[:, 4]) > 0.5
    return abs_coords, pen_up

def plot_sketch5(stroke5_np, title='', color='black', ax=None):
    coords, pen_up = stroke5_to_absolute(stroke5_np)
    if ax is None: fig, ax = plt.subplots(figsize=(3,3))
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off'); ax.set_title(title, fontsize=9)
    start = 0
    for i in range(len(pen_up)):
        if pen_up[i]:
            seg = coords[start:i+1]
            ax.plot(seg[:,0], seg[:,1], color=color, linewidth=1.5)
            start = i+1

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
plot_sketch5(context.squeeze(0).cpu().numpy(), title='Context (input)', color='blue', ax=axes[0])
for i, comp in enumerate(completions):
    plot_sketch5(comp, title=f'Completion {i+1}', ax=axes[i+1])

plt.suptitle('Same context, 4 different completions (temperature=0.8)', fontsize=11)
plt.tight_layout(); plt.show()

# TODO: Try temperature=0.3 and temperature=1.5. How does it affect the completions?
# YOUR CODE HERE


## Checkpoint 4 complete -- Creative Diversity!

The model now generates multiple unique completions for the same starting sketch.
Next: the two assignments to consolidate what you have built.